# Comment Classification: LLM"

Before running this notebook:
1. Install Ollama: https://ollama.com
2. Pull the model: ollama pull qwen3:8b

In [17]:
import pandas as pd
import ollama
import re
from tqdm import tqdm

In [ ]:
# loading comments data
data_path = "data/comments/comments_manual_coding.xlsx"
data_comments = pd.read_excel(data_path)

In [19]:
SYSTEM_PROMPT = """You are a text classifier for a social science study.

Task:
Determine whether the comment contains personal biographical or experiential information.

Facet: Personal experience

Code 1 (Present):
The commenter refers to their own life events, experiences, personal circumstances, or those of close family members (e.g., parents, siblings, spouse, children).

Code 0 (Absent):
The comment contains no personal biographical or experiential information. Opinions, arguments, recommendations, policy preferences, or statements such as "I think", "I believe", or "I support" do NOT count unless they include personal experiences or personal circumstances.

Examples of 1:
- "I witnessed the terrorist attacks in Turkey last year."
- "My brother was near the attack."
- "I've owned firearms for twenty years."
- "My daughter goes to that school."

Examples of 0:
- "Weapons purchases should not only be restricted online."
- "Hire more staff."
- "I think this policy is ineffective."
- "We should ban assault weapons."

Return ONLY 0 or 1. Nothing else."""

def classify_personal_experience(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ],
        options={"temperature": 0}
    )

    out = response["message"]["content"].strip()
    match = re.search(r"\b(0|1)\b", out)
    return int(match.group(1)) if match else -1

In [20]:
texts = data_comments["raw"].fillna("")

texts = texts.head(50)

results = []
for t in tqdm(texts, desc="Classifying"):
    results.append(classify_personal_experience(t))

Classifying: 100%|██████████| 50/50 [12:05<00:00, 14.50s/it]


In [21]:
data_comments = data_comments.head(50)

data_comments["label_pers_exp"] = results
data_comments[["raw", "label_pers_exp"]].head()

<positron-console-cell-21>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


,raw,label_pers_exp
0,Zum Frühwarnsystem: ich wundere mich schon seh...,1
1,Punkt 9 müsste eigentlich ganz vorne kommen. A...,0
2,"[quote=""system, post:1, topic:15""] Konzept [/q...",0
3,Wenn alles so ausgeführt würde wie geschrieben...,0
4,"Erst einmal gut, dass man sich nicht nur auf e...",0


In [22]:
from sklearn.metrics import classification_report, cohen_kappa_score

# ground truth column = pers_exp, predicted = label_pers_exp
y_true = data_comments["pers_exp"]
y_pred = data_comments["label_pers_exp"]

# drop rows where either is missing or prediction failed (-1)
mask = (y_true.notna()) & (y_pred != -1)
y_true = y_true[mask]
y_pred = y_pred[mask]

print(classification_report(y_true, y_pred, target_names=["absent (0)", "present (1)"]))
print(f"Cohen's Kappa: {cohen_kappa_score(y_true, y_pred):.3f}")

              precision    recall  f1-score   support

  absent (0)       0.96      0.98      0.97        45
 present (1)       0.75      0.60      0.67         5

    accuracy                           0.94        50
   macro avg       0.85      0.79      0.82        50
weighted avg       0.94      0.94      0.94        50

Cohen's Kappa: 0.634


In [11]:
data_comments.to_excel("data_comments_classified.xlsx", index=False)

# Claude Sonnet

In [23]:
claude_path = "data/comments/comments_sonnet_coded.xlsx"
data_claude = pd.read_excel(claude_path)

data_claude.head(50)



,coder,post_number,user,topic,raw,pers_exp,emot_exp,pol_opin,breadth,valence,...,created_at,reply_to_post_number,reply_chain,vm,key,value,key...7,value...8,key...9,value...10
0,Claude Sonnet 5 (AI first-pass),7,Sternmarke,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Mir taucht in dem Plan viel zu h‰ufig die Form...,0,0,0,1,2,...,2017-08-29T10:42:34Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
1,Claude Sonnet 5 (AI first-pass),8,UdoUli,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Man sollte nicht die Bundeswehr f¸r die innere...,0,0,1,2,2,...,2017-08-29T11:31:49Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
2,Claude Sonnet 5 (AI first-pass),9,pokerwuerfel,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),"Heisse Luft, wie Alles, was St.Angela Merkel k...",0,1,1,2,2,...,2017-08-29T12:30:05Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
3,Claude Sonnet 5 (AI first-pass),10,Zehle0,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Wenn Sie das Alles machen wollen? Dann mal vie...,0,0,0,0,2,...,2017-08-29T13:42:51Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
4,Claude Sonnet 5 (AI first-pass),11,MrRight,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Der Neun-Punkte-Plan ist so aussagekr‰ftig wie...,0,0,1,3,2,...,2017-08-29T14:09:52Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
5,Claude Sonnet 5 (AI first-pass),12,Felix,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Einige Punkte machen Sinn und sind l‰ngst ¸ber...,0,0,0,0,3,...,2017-08-29T14:27:25Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
6,Claude Sonnet 5 (AI first-pass),13,Maria,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),In der Theorie hˆrt sich das alles recht vern¸...,0,0,1,3,3,...,2017-08-29T14:38:03Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
7,Claude Sonnet 5 (AI first-pass),14,diglage,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),"Im Prinzip ist der Plan ein guter Anfang,allei...",0,0,0,1,3,...,2017-08-29T14:39:48Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN
8,Claude Sonnet 5 (AI first-pass),15,Maria,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),"Das ist unser Problem. "" Schnell "" geht gar ni...",0,0,0,1,3,...,2017-08-29T14:43:24Z,14.0,"Im Prinzip ist der Plan ein guter Anfang,allei...",vm1,NaN,NaN,NaN,NaN,NaN,NaN
9,Claude Sonnet 5 (AI first-pass),16,Tyler,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),"Liest sich ganz gut, ist aber insgesamt viel z...",0,0,0,2,3,...,2017-08-29T15:11:04Z,NaN,NaN,vm1,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# --- merge on post_number so rows line up ---
merged = data_claude.merge(data_comments, on="post_number", suffixes=("_claude", "_manual"))
print(f"Matched {len(merged)} of {len(data_claude)} (Claude) / {len(data_comments)} (manual) rows\n")

variables = {
    "pers_exp": None,      # binary -> unweighted kappa
    "emot_exp": None,      # binary -> unweighted kappa
    "pol_opin": None,      # binary -> unweighted kappa
    "contr":    None,      # binary -> unweighted kappa
    "breadth":  "linear",  # ordinal count (0-3) -> linear-weighted kappa
    "valence":  None,      # 4 unordered-ish categories -> unweighted kappa
}

results = []
for var, weight in variables.items():
    a = merged[f"{var}_claude"]
    b = merged[f"{var}_manual"]

    pct_agree = (a == b).mean()
    kappa = cohen_kappa_score(a, b, weights=weight)
    results.append({"variable": var, "pct_agreement": round(pct_agree, 3),
                     "cohens_kappa": round(kappa, 3), "weighted": weight or "none"})

    print(f"--- {var} ---")
    print(f"Percent agreement: {pct_agree:.1%}   Cohen's kappa: {kappa:.3f}")
    labels = sorted(set(a) | set(b))
    cm = pd.DataFrame(confusion_matrix(a, b, labels=labels), index=labels, columns=labels)
    cm.index.name, cm.columns.name = "Claude", "manual"
    print(cm, "\n")

summary = pd.DataFrame(results)
print("=== Summary ===")
print(summary)

Matched 42 of 149 (Claude) / 50 (manual) rows

--- pers_exp ---
Percent agreement: 88.1%   Cohen's kappa: 0.000
manual   0  1
Claude       
0       37  5
1        0  0 

--- emot_exp ---
Percent agreement: 78.6%   Cohen's kappa: 0.488
manual   0  1
Claude       
0       25  5
1        4  8 

--- pol_opin ---
Percent agreement: 59.5%   Cohen's kappa: 0.236
manual   0   1
Claude        
0        7   1
1       16  18 

--- contr ---
Percent agreement: 64.3%   Cohen's kappa: 0.351
manual   0   1
Claude        
0       10   0
1       15  17 

--- breadth ---
Percent agreement: 31.0%   Cohen's kappa: 0.250
manual  0  1  2  3  4  5  6
Claude                     
0       1  0  0  0  0  0  0
1       5  8  3  1  0  0  0
2       2  4  2  1  1  0  0
3       1  3  5  2  1  1  1
4       0  0  0  0  0  0  0
5       0  0  0  0  0  0  0
6       0  0  0  0  0  0  0 

--- valence ---
Percent agreement: 50.0%   Cohen's kappa: 0.297
manual  1   2  3  4
Claude             
1       1   1  1  2
2       1  14 